In [0]:
# Notebook: 02_silver/01_dim_team_fact_game.py
# Project : nba-datalakehouse
# Purpose : Build the first Silver tables from Bronze:
#             1) workspace.nba_silver.dim_team
#             2) workspace.nba_silver.fact_game
#
# Why start here?
# --------------
# - Teams + Games are the "spine" of basketball analytics.
# - They are relatively flat (easy to type + validate).
# - They become the backbone for odds joins and (later) player stats joins.
#
# SILVER CLEANING STANDARD (what we do consistently)
# --------------------------------------------------
# For each Silver table:
#   A) Select the fields we care about (explicit schema)
#   B) Cast to correct types (ints, dates, timestamps)
#   C) Standardize / trim strings where useful
#   D) Enforce grain by deduplicating on the natural key
#   E) Keep audit columns so we can trace provenance:
#        - source_dt (date partition from Bronze)
#        - ingested_at (from Bronze metadata if available)
#
# Notes about your Bronze structure:
# ---------------------------------
# Your Bronze Delta tables were written from JSONL as:
#   - dt   : top-level partition column (string/date-like)
#   - meta : struct (contains dt and ingestion metadata)
#   - data : struct (raw API payload)
#
# We'll read from:
#   - workspace.nba_bronze.balldontlie_teams
#   - workspace.nba_bronze.balldontlie_games
# And write to:
#   - workspace.nba_silver.dim_team
#   - workspace.nba_silver.fact_game

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
BRONZE_TEAMS = "workspace.nba_bronze.balldontlie_teams"
BRONZE_GAMES = "workspace.nba_bronze.balldontlie_games"

SILVER_DIM_TEAM = "workspace.nba_silver.dim_team"
SILVER_FACT_GAME = "workspace.nba_silver.fact_game"

In [0]:
# -----------------------------------------------------------------------------
# 0) Read Bronze tables
# -----------------------------------------------------------------------------
teams_bronze = spark.table(BRONZE_TEAMS)
games_bronze = spark.table(BRONZE_GAMES)

print("Bronze teams rows:", teams_bronze.count())
print("Bronze games rows:", games_bronze.count())

In [0]:
# -----------------------------------------------------------------------------
# 1) Build Silver dim_team
# -----------------------------------------------------------------------------
# Grain (intended): 1 row per BallDontLie team id (team_id_bdl)
#
# Cleaning logic:
# - Cast ids to INT
# - Trim strings (protect against accidental whitespace)
# - Deduplicate on team_id_bdl:
#     If duplicates exist, keep the newest row by ingested_at (if present),
#     otherwise keep an arbitrary row (rare for teams).
#
# Audit columns:
# - source_dt: DATE derived from bronze dt
# - ingested_at: TIMESTAMP from bronze meta if present (fallback to current_timestamp)

def _safe_ingested_at(df):
    """Return a column expression for ingested_at, falling back to current_timestamp."""
    # meta.ingested_at may or may not exist depending on your ingestion metadata.
    return F.coalesce(F.col("meta.ingested_at").cast("timestamp"), F.current_timestamp())

dim_team_raw = (
    teams_bronze
    .select(
        F.col("data.id").cast("int").alias("team_id_bdl"),
        F.trim(F.col("data.abbreviation")).alias("abbreviation"),
        F.trim(F.col("data.city")).alias("city"),
        F.trim(F.col("data.conference")).alias("conference"),
        F.trim(F.col("data.division")).alias("division"),
        F.trim(F.col("data.full_name")).alias("full_name"),
        F.trim(F.col("data.name")).alias("name"),
        F.col("dt").cast("date").alias("source_dt"),
        _safe_ingested_at(teams_bronze).alias("ingested_at"),
        F.lit("balldontlie").alias("source_system"),
    )
)

# Basic “bad key” filter: if team_id is null, we cannot use the row.
dim_team_raw = dim_team_raw.filter(F.col("team_id_bdl").isNotNull())

# Deduplicate: keep the most recent by ingested_at
w_team = Window.partitionBy("team_id_bdl").orderBy(F.col("ingested_at").desc())
dim_team = (
    dim_team_raw
    .withColumn("_rn", F.row_number().over(w_team))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# Write Silver dim_team
(dim_team.write
 .format("delta")
 .mode("overwrite")   # v1: rebuild full dimension; teams are tiny and stable
 .saveAsTable(SILVER_DIM_TEAM))

print("Wrote:", SILVER_DIM_TEAM, "rows=", dim_team.count())
display(dim_team.orderBy("team_id_bdl").limit(10))


In [0]:
# -----------------------------------------------------------------------------
# 2) Build Silver fact_game
# -----------------------------------------------------------------------------
# Grain (intended): 1 row per BallDontLie game id (game_id_bdl)
#
# Cleaning logic:
# - Cast ids/scores to INT
# - Parse dates/timestamps
# - Keep only rows with non-null game_id
# - Deduplicate on game_id_bdl (keep newest by ingested_at)
# - Minimal sanity guards:
#     - negative scores -> set to null (should never happen, but safe)
#
# Audit columns:
# - source_dt: DATE derived from bronze dt
# - ingested_at: TIMESTAMP from bronze meta if present (fallback to current_timestamp)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"DROP TABLE IF EXISTS {SILVER_FACT_GAME}")

def _ingested_at():
    return F.coalesce(
        F.to_timestamp(F.col("meta.ingested_at")),
        F.current_timestamp(),
    )

def _start_time_utc():
    return F.coalesce(
        F.to_timestamp(F.col("datetime")),
        F.to_timestamp(F.col("date")),
        F.to_timestamp(F.col("data.datetime")),
        F.to_timestamp(F.col("data.date")),
    )

def _game_date_utc():
    return F.to_date(_start_time_utc())

def _is_final_status():
    s = F.lower(F.trim(F.col("status")))
    return s.startswith("final")

fact_game_raw = (
    games_bronze
    .select(
        F.col("id").cast("int").alias("game_id_bdl"),
        _start_time_utc().alias("start_time_utc"),
        _game_date_utc().alias("game_date_utc"),
        F.col("season").cast("int").alias("season_start_year"),
        F.trim(F.col("status")).alias("status"),
        F.col("home_team.id").cast("int").alias("home_team_id_bdl"),
        F.col("visitor_team.id").cast("int").alias("visitor_team_id_bdl"),
        F.col("home_team_score").cast("int").alias("home_score_raw"),
        F.col("visitor_team_score").cast("int").alias("visitor_score_raw"),
        F.coalesce(F.col("dt"), F.col("meta.dt")).cast("date").alias("source_dt"),
        _ingested_at().alias("ingested_at"),
        F.lit("balldontlie").alias("source_system"),
    )
    .filter(F.col("game_id_bdl").isNotNull())
)

is_final = _is_final_status()

fact_game = (
    fact_game_raw
    .withColumn("home_score", F.when(is_final, F.when(F.col("home_score_raw") >= 0, F.col("home_score_raw"))).otherwise(F.lit(None).cast("int")))
    .withColumn("visitor_score", F.when(is_final, F.when(F.col("visitor_score_raw") >= 0, F.col("visitor_score_raw"))).otherwise(F.lit(None).cast("int")))
    .drop("home_score_raw", "visitor_score_raw")
)

w_game = Window.partitionBy("game_id_bdl").orderBy(F.col("ingested_at").desc())
fact_game = (
    fact_game
    .withColumn("_rn", F.row_number().over(w_game))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

(fact_game.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("source_dt")
 .saveAsTable(SILVER_FACT_GAME))

print("Wrote:", SILVER_FACT_GAME, "rows=", fact_game.count())
display(fact_game.orderBy(F.col("game_date_utc").desc(), F.col("start_time_utc").desc()).limit(20))


In [0]:
games_bronze.printSchema()


In [0]:
# -----------------------------------------------------------------------------
# 3) Quick sanity checks (Python)
# -----------------------------------------------------------------------------
# These are not a replacement for a validation SQL notebook, but they catch obvious issues quickly.

display(
    fact_game
    .filter(F.col("season_start_year") == 2025)
    .groupBy("season_start_year")
    .agg(
        F.min("game_date_utc").alias("min_game_date"),
        F.max("game_date_utc").alias("max_game_date"),
        F.count("*").alias("n_games"),
    )
)


In [0]:
# =====================================================================================
# Success criteria
# ----------------
# - Silver tables exist:
#     workspace.nba_silver.dim_team
#     workspace.nba_silver.fact_game
# - Row counts are > 0
# - Keys are non-null
# - Types are correct (team/game IDs are INT, dates are DATE, timestamps are TIMESTAMP)
#
# Next notebook:
#   99_validation/silver_checks.sql  (we will create this after odds tables too)
# Next build step:
#   02_silver/02_odds_event_bookmaker_market_outcome.py
# =====================================================================================
